# Sliced Normal

This is the original repository workflow using the cached implementation in
`sliced_normals/`: data → FMLE → fit → approximate likelihood → MCMC → plots.

Run the first code cell in your current kernel to create `.venv`, install the
requirements, and register **Python (Sliced Normal)**. Select that kernel, then
run all cells. Python 3.11 is required for this setup cell. It uses standard
library modules and does not clone or delete a repository.

The sampler retains unbounded support while fitting normalizes on a box.
Sampling plots are exploratory and do not establish convergence.

The final section separately demonstrates primal SN fitting: extract an FMLE
sum-of-squares basis, optimize its coefficients, inspect diagnostics, and compare
likelihoods and generated samples with its matched FMLE initialization.

In [ ]:
# Run this cell first. It does not require NumPy, pandas, or the fitting packages.
import os
import sys
import json
import shlex
import shutil
import hashlib
import subprocess
from pathlib import Path

BASE_PYTHON = None  # Optional: '/absolute/path/to/python3.11'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sliced_normals' / 'optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or one of its subfolders.')
ENV_DIR = ROOT / '.venv'
ENV_PYTHON = ENV_DIR / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
REQUIREMENTS = ROOT / 'requirements.txt'
KERNEL_NAME = 'sliced-normal'
KERNEL_LABEL = 'Python (Sliced Normal)'

def run_setup_command(args):
    args = [str(arg) for arg in args]
    print('>', shlex.join(args), flush=True)
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f'Setup command failed with exit code {status}. See the output above.')

def python_version(executable):
    try:
        result = subprocess.run(
            [str(executable), '-c', 'import sys,json; print(json.dumps(list(sys.version_info[:2])))'],
            check=True, capture_output=True, text=True)
        return tuple(json.loads(result.stdout))
    except (OSError, subprocess.SubprocessError, ValueError):
        return None

if not REQUIREMENTS.is_file():
    raise RuntimeError(f'Missing dependency file: {REQUIREMENTS}')
if not ENV_PYTHON.is_file():
    candidates = ([BASE_PYTHON] if BASE_PYTHON else
                  [shutil.which('python3.11'), sys.executable, shutil.which('python3')])
    base = next((candidate for candidate in candidates
                 if candidate and python_version(candidate) == (3, 11)), None)
    if base is None:
        raise RuntimeError('Install Python 3.11, set BASE_PYTHON above to its executable, and rerun this cell.')
    run_setup_command([base, '-m', 'venv', ENV_DIR])
if python_version(ENV_PYTHON) != (3, 11):
    raise RuntimeError(f'{ENV_DIR} is not a working Python 3.11 environment. '
                       'Choose a different ENV_DIR or recreate that environment, then rerun setup.')

run_setup_command([ENV_PYTHON, '-m', 'pip', '--require-virtualenv', 'install', '-r', REQUIREMENTS])
run_setup_command([ENV_PYTHON, '-m', 'pip', '--require-virtualenv', 'install', '-e', ROOT / 'sliced_normals'])
run_setup_command([ENV_PYTHON, '-m', 'pip', 'check'])
run_setup_command([
    ENV_PYTHON, '-c',
    'import sys; sys.path.insert(0, sys.argv[1]); '
    'import numpy,pandas,scipy,autograd,pymanopt,emcee,matplotlib,threadpoolctl,ipykernel; '
    'import sliced_normals; '
    'print("All notebook and fitter imports passed.")', str(ROOT)])
run_setup_command([ENV_PYTHON, '-m', 'ipykernel', 'install', '--user',
                   '--name', KERNEL_NAME, '--display-name', KERNEL_LABEL])

print('\nEnvironment:', ENV_DIR)
print('Current kernel:', sys.executable)
if Path(sys.prefix).resolve() == ENV_DIR.resolve():
    print('The notebook is already using the new environment. Continue with the next cell.')
else:
    print(f'NEXT: select "{KERNEL_LABEL}" in the notebook kernel selector, then run all cells.')
    print('If it is not listed, refresh the kernel list or choose this interpreter:', ENV_PYTHON)
if os.name != 'nt':
    print('Optional terminal activation: source ' + shlex.quote(str(ENV_DIR / 'bin/activate')))
else:
    print('Optional PowerShell activation: & "' + str(ENV_DIR / 'Scripts/Activate.ps1') + '"')

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'sliced_normals/optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or Demo folder.')
if Path(sys.prefix).resolve() != (ROOT/'.venv').resolve():
    raise RuntimeError('Select the Python (Sliced Normal) kernel created by the setup cell, then run all cells.')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pymanopt.manifolds import SymmetricPositiveDefinite
from sliced_normals import (get_FMLE, estimate_optimal_B, evaluate_true_log_likelihood,
                            run_mcmc_sampler, get_F_and_Random_Samples, CachedObjective,
                            estimate_primal_B)
print('Python:', sys.executable)


In [ ]:
DEGREE = 2
GRID = 5000
MAX_ITERATIONS = 2000
GRADIENT_TOLERANCE = 5e-4
EVALUATION_GRID = 50000
STEPS = 2000
BURNIN = 300
THIN = 5
SEED = 42


## Data and FMLE

In [ ]:
data = pd.read_csv(ROOT/'sliced_normals/Sample_Data.csv')[['X_1','X_2','X_3']]
np.random.seed(SEED)
B_fmle = get_FMLE(data, DEGREE)
print('Observations:', len(data))
sns.pairplot(data, plot_kws={'s': 5})
plt.show()


## Maximum-likelihood fitting
The matrix fitter uses Pymanopt steepest descent with the default partition
floor. This cell independently checks the Riemannian gradient at its returned
matrix. An iteration limit or tiny-step stop can leave the requested tolerance
unmet. The coefficient-only primal fitter is demonstrated in a separate section
below.


In [ ]:
np.random.seed(SEED)
B_opt = estimate_optimal_B(data, degree=DEGREE, n_grid=GRID,
                           max_iterations=MAX_ITERATIONS,
                           min_gradient_norm=GRADIENT_TOLERANCE, verbosity=1)
np.random.seed(SEED)
Z, Z_grid, volume = get_F_and_Random_Samples(data, DEGREE, GRID)
objective = CachedObjective(Z.to_numpy(), Z_grid.to_numpy(), volume)
manifold = SymmetricPositiveDefinite(len(B_opt))
rgrad = manifold.euclidean_to_riemannian_gradient(B_opt, objective.gradient(B_opt))
final_gradient_norm = float(manifold.norm(B_opt, rgrad))
print('Final objective:', objective.cost(B_opt))
print('Final gradient norm:', final_gradient_norm)
print('Requested gradient tolerance met:', final_gradient_norm < GRADIENT_TOLERANCE)


## Independent normalization check
These are approximate in-sample mean log likelihoods. Both models use identical fresh integration points on the same data-derived support. This is not held-out validation.

In [ ]:
for name, B in [('FMLE', B_fmle), ('Fitted model', B_opt)]:
    np.random.seed(SEED + 1000)
    value = evaluate_true_log_likelihood(data, B, DEGREE, n_grid=EVALUATION_GRID)
    print(name, 'approximate mean log likelihood:', float(value))


## MCMC sampling
STEPS counts steps per walker. Check chain mixing, effective sample size, and mode coverage separately.

In [ ]:
np.random.seed(SEED + 2000)
samples_opt = run_mcmc_sampler(data, B_opt, d=DEGREE, num_samples=STEPS,
                               burnin=BURNIN, thin=THIN)
np.random.seed(SEED + 2001)
samples_fmle = run_mcmc_sampler(data, B_fmle, d=DEGREE, num_samples=STEPS,
                                burnin=BURNIN, thin=THIN)
print('Retained rows:', len(samples_opt), len(samples_fmle))


## Pairplots
Use common coordinate limits to compare observed data and retained samples.

In [ ]:
frames = [('Data', data), ('Fitted model samples', samples_opt), ('FMLE samples', samples_fmle)]
limits = {column: (min(frame[column].min() for _, frame in frames),
                   max(frame[column].max() for _, frame in frames)) for column in data.columns}
for name, frame in frames:
    plot = sns.pairplot(frame, plot_kws={'s': 5})
    plot.fig.suptitle(name, y=1.02)
    for i, y in enumerate(data.columns):
        for j, x in enumerate(data.columns):
            plot.axes[i,j].set_xlim(limits[x])
            if i != j:
                plot.axes[i,j].set_ylim(limits[y])
    plt.show()


## Primal SN: FMLE sum-of-squares coefficient fitting
This section uses the shared setup, settings, and `data` loaded above. It has its
own `B_primal`, diagnostics, and samples. You can run it after the data cell
without running the matrix-fitting sections.

The weighted FMLE supplies fixed squared polynomials `h_j(x)**2`. Only their
positive coefficients are optimized; all coefficients start at one. For this
three-variable, degree-2 example there are nine coefficients, compared with
55 raw symmetric matrix entries. The fixed-basis objective is convex and uses
SciPy L-BFGS-B without a partition floor. It is a restricted SN fit.

`PRIMAL_WEIGHTS = None` uses equal weights. To demonstrate weighted fitting,
supply one nonnegative weight per row of `data`; the comparison below uses
those same weights. Its matched FMLE initializer is `primal_info.initial_B`,
which uses weighted ML covariance and feature-scaled regularization, rather
than the legacy FMLE covariance convention.


In [ ]:
PRIMAL_WEIGHTS = None  # Optional array of length len(data)
PRIMAL_BASIS = "eigen"  # Or "cholesky"
PRIMAL_MIN_COEFFICIENT = 1e-6

np.random.seed(SEED)
B_primal, primal_info = estimate_primal_B(
    data, degree=DEGREE, n_grid=GRID, sample_weight=PRIMAL_WEIGHTS,
    basis=PRIMAL_BASIS, min_coefficient=PRIMAL_MIN_COEFFICIENT,
    max_iterations=MAX_ITERATIONS, min_gradient_norm=GRADIENT_TOLERANCE,
    return_info=True, verbosity=0)

print('Number of optimized coefficients:', len(primal_info.coefficients))
print('Initial objective:', primal_info.initial_fun)
print('Final objective:', primal_info.fun)
print('Final projected gradient norm:', primal_info.projected_gradient_norm)
print('Requested gradient tolerance met:', primal_info.gradient_tolerance_met)
print('Stopping reason:', primal_info.message)
print('Input-weight effective sample size:', primal_info.effective_sample_size)
print('Fitting-grid effective sample size:', primal_info.integration_ess)

primal_coefficients = pd.DataFrame({
    'squared_polynomial': np.arange(1, len(primal_info.coefficients) + 1),
    'initial_coefficient': np.ones(len(primal_info.coefficients)),
    'fitted_coefficient': primal_info.coefficients,
    'at_lower_bound': np.isclose(primal_info.coefficients, PRIMAL_MIN_COEFFICIENT,
                                 rtol=0, atol=1e-12),
})
print(primal_coefficients.to_string(index=False))


### Coefficients and independent normalization check
The projected gradient accounts for coefficient bounds; it is different from
the full matrix gradient. Solver success alone does not establish the requested
tolerance. The fitting objective uses a fixed grid, so check normalization
independently even when that tolerance is met. A grid that makes the numerical
likelihood demonstrably unbounded raises an error instead of returning a model.

The comparison uses identical fresh integration points for the matched FMLE
initializer and primal fit, on the same support and with the same sample weights.
These are in-sample mean log likelihoods, not held-out validation. Neither the
fitting-grid ESS nor the fresh-grid ESS certifies integration accuracy.


In [ ]:
from scipy.special import logsumexp

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(primal_coefficients['squared_polynomial'], primal_info.coefficients)
ax.axhline(1, color='black', linestyle='--', label='FMLE coefficients')
ax.set_xlabel('Squared polynomial')
ax.set_ylabel('Coefficient')
ax.set_title('Primal SN coefficient optimization')
ax.legend()
plt.show()

np.random.seed(SEED + 1000)
Z_check, grid_check, check_volume = get_F_and_Random_Samples(
    data, DEGREE, EVALUATION_GRID)
Z_check = Z_check.to_numpy()
grid_check = grid_check.to_numpy()
if PRIMAL_WEIGHTS is None:
    check_weights = np.full(len(data), 1 / len(data))
else:
    check_weights = np.asarray(PRIMAL_WEIGHTS, dtype=float)
    check_weights = check_weights / check_weights.max()
    check_weights = check_weights / check_weights.sum()

primal_comparison_rows = []
for name, B in [('Primal FMLE initialization', primal_info.initial_B),
                ('Primal fitted model', B_primal)]:
    data_energy = np.einsum('ij,jk,ik->i', Z_check, B, Z_check)
    grid_energy = np.einsum('ij,jk,ik->i', grid_check, B, grid_check)
    log_grid_sum = logsumexp(-grid_energy)
    log_partition = log_grid_sum + np.log(check_volume / len(grid_check))
    grid_probabilities = np.exp(-grid_energy - log_grid_sum)
    primal_comparison_rows.append({
        'model': name,
        'weighted_mean_log_likelihood': float(-check_weights @ data_energy - log_partition),
        'fresh_grid_ESS': float(1 / (grid_probabilities @ grid_probabilities)),
    })
primal_comparison = pd.DataFrame(primal_comparison_rows)
print(primal_comparison.to_string(index=False))


### Samples from the primal fit and its FMLE initialization
Both matrices use the existing MCMC sampler. Compare generated samples with the
weighted data target; the target pairplots below are produced by resampling the
data according to its weights. With equal weights, the original data is shown.
Resampled points are for plotting only and are not used to fit the model.

The sampler is unbounded while fitting normalizes on a box. These plots are
exploratory; inspect chain mixing and mode coverage separately.


In [ ]:
np.random.seed(SEED + 3000)
samples_primal = run_mcmc_sampler(
    data, B_primal, d=DEGREE, num_samples=STEPS, burnin=BURNIN, thin=THIN)
np.random.seed(SEED + 3001)
samples_primal_fmle = run_mcmc_sampler(
    data, primal_info.initial_B, d=DEGREE, num_samples=STEPS,
    burnin=BURNIN, thin=THIN)
if PRIMAL_WEIGHTS is None:
    primal_target = data
else:
    plot_rng = np.random.default_rng(SEED + 3002)
    target_indices = plot_rng.choice(len(data), size=len(data), p=check_weights)
    primal_target = data.iloc[target_indices].reset_index(drop=True)

primal_frames = [('Primal target data', primal_target),
                 ('Primal fitted samples', samples_primal),
                 ('Primal FMLE samples', samples_primal_fmle)]
primal_limits = {
    column: (min(frame[column].min() for _, frame in primal_frames),
             max(frame[column].max() for _, frame in primal_frames))
    for column in data.columns
}
for name, frame in primal_frames:
    plot = sns.pairplot(frame, plot_kws={'s': 5})
    plot.fig.suptitle(name, y=1.02)
    for i, y in enumerate(data.columns):
        for j, x in enumerate(data.columns):
            plot.axes[i,j].set_xlim(primal_limits[x])
            if i != j:
                plot.axes[i,j].set_ylim(primal_limits[y])
    plt.show()
print('Retained primal/FMLE sample rows:', len(samples_primal), len(samples_primal_fmle))


## Weighted fitting and comparison
Use this section when each observation has a sample weight. It uses the shared
setup/settings and `data` above, and can run immediately after the data cell.
The illustrative weights below smoothly change the importance of observations;
replace `demo_weights` with your actual weights in the same row order.

The same weights enter FMLE initialization, full matrix fitting, primal fitting,
and fresh-grid likelihood evaluation. Weights are normalized internally, so
multiplying all of them by a common positive constant does not change the target.
Zero weights are allowed, but at least one weight must be positive. Weights are
positional and do not change the integration box or integration-point weights.

Explicitly weighted FMLE uses ML covariance moments; the legacy unweighted
FMLE above retains its historical `n-1` covariance. The primal fit also applies
its feature-scaled covariance regularization.


In [ ]:
# Illustrative weights only: replace this array with your actual sample weights.
weight_coordinate = data.iloc[:, 0].to_numpy(dtype=float)
weight_coordinate = (weight_coordinate - weight_coordinate.mean()) / weight_coordinate.std()
demo_weights = 1.0 + 0.5 * np.tanh(weight_coordinate)
normalized_demo_weights = demo_weights / demo_weights.sum()
print('Input-weight effective sample size:', 1 / np.sum(normalized_demo_weights**2))

B_weighted_fmle = get_FMLE(data, d=DEGREE, sample_weight=demo_weights)
np.random.seed(SEED + 4000)
B_weighted_full = estimate_optimal_B(
    data, degree=DEGREE, n_grid=GRID, sample_weight=demo_weights,
    max_iterations=MAX_ITERATIONS, min_gradient_norm=GRADIENT_TOLERANCE,
    verbosity=1)
np.random.seed(SEED + 4000)
B_weighted_primal, weighted_primal_info = estimate_primal_B(
    data, degree=DEGREE, n_grid=GRID, sample_weight=demo_weights,
    max_iterations=MAX_ITERATIONS, min_gradient_norm=GRADIENT_TOLERANCE,
    return_info=True, verbosity=1)
print('Primal coefficient count:', len(weighted_primal_info.coefficients))


### Fresh-grid weighted likelihood comparison
Higher weighted mean log likelihood is better. These are in-sample scores with
independent numerical normalization, not held-out validation. Each model below
uses the same fresh integration points and sample weights. Matrix and primal
stopping criteria differ, so this table is not an equal-accuracy timing test.

Sampling from either fitted model uses its `B` matrix directly. The sample
weights have already entered the fit; do not reweight generated samples again.


In [ ]:
weighted_models = [('Weighted FMLE', B_weighted_fmle),
                   ('Weighted full matrix', B_weighted_full),
                   ('Weighted primal SN', B_weighted_primal)]
weighted_comparison_rows = []
for name, B in weighted_models:
    np.random.seed(SEED + 5000)
    score = evaluate_true_log_likelihood(
        data, B, d=DEGREE, n_grid=EVALUATION_GRID, sample_weight=demo_weights)
    weighted_comparison_rows.append({
        'model': name, 'weighted_mean_log_likelihood': float(score)})
weighted_comparison = pd.DataFrame(weighted_comparison_rows)
print(weighted_comparison.to_string(index=False))


## Adaptive primal SN: equal and nonuniform weights

The adaptive fitter starts with the fixed-basis primal model and proposes additional
squared polynomial combinations. Rejected batches are retried individually; every
accepted refit adjusts all coefficients. The same API handles equal and nonuniform
observation weights. The weighted run below uses illustrative weights; replace them
with your row-aligned observation weights.

This section can run after the shared setup, settings and data cells. It uses
independent integration draws for screening and final audits, and does not reuse
fitted objects from earlier sections. The small addition cap limits this example's
cost. Integration diagnostics or marginal checks may stop the search early.
Uniform box integration can be inadequate for narrow data; supply a known-density
`integration_sampler` when needed. Optimization convergence and integration accuracy
are separate questions.


In [ ]:
from sliced_normals import estimate_adaptive_primal_B

adaptive_coordinate = data.iloc[:, 0].to_numpy(dtype=float)
adaptive_coordinate = (adaptive_coordinate-adaptive_coordinate.mean())/adaptive_coordinate.std()
adaptive_demo_weights = 1.0+0.5*np.tanh(adaptive_coordinate)
adaptive_fits = {}
adaptive_summary = []
for label, observation_weights in [('Equal weights', None), ('Weighted', adaptive_demo_weights)]:
    B_adaptive, adaptive_info = estimate_adaptive_primal_B(
        data, degree=DEGREE, sample_weight=observation_weights,
        n_grid=GRID, n_screen=GRID, n_audit=EVALUATION_GRID,
        max_iterations=MAX_ITERATIONS, min_gradient_norm=GRADIENT_TOLERANCE,
        max_added=5, add_per_round=2, max_rounds=5,
        random_state=SEED+6000, return_info=True, verbosity=1,
    )
    adaptive_fits[label] = (B_adaptive, adaptive_info)
    adaptive_summary.append({
        'weighting':label, 'accepted_additions':adaptive_info.accepted_additions,
        'total_coefficients':len(adaptive_info.coefficients),
        'baseline_training_NLL':adaptive_info.baseline_info.fun,
        'adaptive_training_NLL':adaptive_info.fun,
        'projected_gradient':adaptive_info.projected_gradient_norm,
        'coefficient_tolerance_met':adaptive_info.gradient_tolerance_met,
        'stop':adaptive_info.search_stopping_reason,
    })
print(pd.DataFrame(adaptive_summary).to_string(index=False))


### Independent final audit and attempt history

Each audit batch compares the baseline and final adaptive fit on the same fresh
draws. Likelihoods use the corresponding run's observation weights. Batch variation
reflects numerical integration uncertainty; these are in-sample scores, not
held-out predictions. Scores from different weighting schemes describe different
targets and should not be ranked against each other.

`info.history` records batch and individual attempts and their rejection reasons.
The returned `B` and coefficient diagnostics always belong to the last accepted
model. The marginal guard compares weighted empirical CDFs with both the original
baseline and previous accepted model; it does not fix marginals exactly.


In [ ]:
for label, (B_adaptive, adaptive_info) in adaptive_fits.items():
    audit_rows = [{
        'batch':row['batch'],
        'baseline_weighted_mean_LL':row['baseline']['mean_log_likelihood'],
        'adaptive_weighted_mean_LL':row['adaptive']['mean_log_likelihood'],
        'paired_LL_gain':row['likelihood_gain'],
        'baseline_integration_ESS':row['baseline']['integration_ess'],
        'adaptive_integration_ESS':row['adaptive']['integration_ess'],
    } for row in adaptive_info.audit]
    print('\n'+label+': independent final audit')
    print(pd.DataFrame(audit_rows).to_string(index=False))
    print('Attempt history:')
    print(pd.DataFrame(adaptive_info.history).to_string(index=False)
          if adaptive_info.history else adaptive_info.search_stopping_reason)
